# Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

N_RECORDS = 20000
N_VEHICLES = 2500
START_YEAR = 2015
END_YEAR = 2025

np.random.seed(42)
random.seed(42)

# Static Data

In [ ]:
nissan_models = {
    "Altima": {"engine": "I4", "base_failure_rate": 0.02},
    "Sunny": {"engine": "I4", "base_failure_rate": 0.015},
    "Patrol": {"engine": "V6", "base_failure_rate": 0.035},
    "X-Trail": {"engine": "I4", "base_failure_rate": 0.025},
    "Maxima": {"engine": "V6", "base_failure_rate": 0.03}
}

failure_types = [
    "Engine Overheating",
    "Brake Failure",
    "Battery Failure",
    "Transmission Issue",
    "Suspension Wear"
]

parts_map = {
    "Engine Overheating": ["Radiator", "Coolant Pump"],
    "Brake Failure": ["Brake Pads", "Brake Disc"],
    "Battery Failure": ["Battery"],
    "Transmission Issue": ["Clutch", "Gearbox"],
    "Suspension Wear": ["Shock Absorber", "Control Arm"]
}

seasons = ["Winter", "Spring", "Summer", "Autumn"]

# Helper Functions

In [ ]:
def random_date(start_year, end_year):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))


def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


def temperature_by_season(season):
    if season == "Summer":
        return np.random.normal(42, 5)
    elif season == "Winter":
        return np.random.normal(22, 4)
    elif season == "Spring":
        return np.random.normal(30, 4)
    else:
        return np.random.normal(32, 4)

# Vehicle Registry

In [ ]:
vehicle_registry = {}

for vid in range(1, N_VEHICLES + 1):
    vehicle_id = f"V{vid}"
    
    model = random.choice(list(nissan_models.keys()))
    engine = nissan_models[model]["engine"]
    base_failure_rate = nissan_models[model]["base_failure_rate"]
    
    manufacture_year = random.randint(2008, 2022)
    
    driving_style = random.choices(
        ["Calm", "Moderate", "Aggressive"],
        weights=[0.4, 0.4, 0.2]
    )[0]

    avg_daily_km = max(5, np.random.normal(40, 10))
    
    vehicle_registry[vehicle_id] = {
        "model": model,
        "engine": engine,
        "base_failure_rate": base_failure_rate,
        "manufacture_year": manufacture_year,
        "driving_style": driving_style,
        "avg_daily_km": avg_daily_km
    }

# Generator

In [ ]:
records = []

for i in range(N_RECORDS):

    vehicle_id = f"V{random.randint(1, N_VEHICLES)}"
    v = vehicle_registry[vehicle_id]  # pull consistent vehicle

    model = v["model"]
    engine = v["engine"]
    base_failure_rate = v["base_failure_rate"]
    
    manufacture_year = v["manufacture_year"]
    
    driving_style = v["driving_style"]
    
    service_date = random_date(manufacture_year, END_YEAR)
    season = get_season(service_date)
    temp = temperature_by_season(season)
    
    vehicle_age = service_date.year - manufacture_year
    
    avg_daily_km = v["avg_daily_km"]
    mileage = max(0, int(
        avg_daily_km * 365 * vehicle_age *
        np.random.normal(1.0, 0.05)
    ))
    
    # Failure probability model
    failure_prob = base_failure_rate
    failure_prob += vehicle_age * 0.005
    failure_prob += mileage / 200000    
    
    if season == "Summer":
        failure_prob += 0.05
    
    if driving_style == "Aggressive":
        failure_prob += 0.05

    failure_prob = min(failure_prob, 0.99)
    failure_occurred = np.random.rand() < failure_prob
    
    if failure_occurred:
        failure_type = random.choice(failure_types)
        service_type = random.choices(
            ["Corrective", "Emergency"],
            weights=[0.6, 0.4]
        )[0]
        parts = parts_map[failure_type]
        cost = np.random.normal(400, 150)
        time_to_next_failure = round(np.random.exponential(180))
    else:
        failure_type = "None"
        service_type = "Preventive"
        parts = ["Oil Filter", "Engine Oil"]
        cost = np.random.normal(120, 40)
        time_to_next_failure = round(np.random.exponential(365))
    
    records.append({
        "vehicle_id": vehicle_id,
        "model": model,
        "engine_type": engine,
        "manufacture_year": manufacture_year,
        "service_date": service_date,
        "season": season,
        "ambient_temp": temp,
        "vehicle_age": vehicle_age,
        "mileage": mileage,
        "avg_daily_km": avg_daily_km,
        "driving_style": driving_style,
        "service_type": service_type,
        "failure_occurred": int(failure_occurred),
        "failure_type": failure_type,
        "parts_replaced": ",".join(parts),
        "service_cost": max(20, cost),
        "time_to_next_failure_days": time_to_next_failure
    })

df = pd.DataFrame(records)
df.head()

# Save

In [ ]:
df.to_csv("synthetic_nissan_pdm.csv", index=False)

# Analysis

In [ ]:
df = pd.read_csv("synthetic_nissan_pdm.csv")

In [ ]:
df[(df["vehicle_id"].duplicated(keep=False)) & (df["vehicle_id"] == "V457")]